# Comparing Scenarios on premise Data

`TimexLCASettings` holds everything one calculation needs - the demand, the
method, the background selection, and every timeline/LCI/LCIA option - so one
object is also the record of what was run. `TimexLCA(...).run()`
executes it, `run()` can be called again with overrides, and
`TimexLCA.compare()` runs a list of them into one table.

We follow that through on a real prospective background, from an empty
project to a scenario comparison: build ecoinvent 3.12 (cutoff) plus REMIND-EU
SSP2-NDC databases for 2020-2100 with
[premise](https://github.com/polca/premise), point a small electric-vehicle
foreground at them, and compare what an EV costs depending on the year it is
bought.

> The build step below is shown rather than re-run each time - it needs an
> ecoinvent licence and a premise key, and takes tens of minutes. Its log
> output is reproduced as you would see it. Everything after it is executed,
> against a project (`ei312_REMIND_EU`) that already went through it.

## Building the background databases

`bw_timex` ships no data of its own. Given a `scenario` that describes what
you want, [PR #222](https://github.com/brightway-lca/bw_timex/pull/222) builds
whatever the project is missing with premise, importing ecoinvent first if it
isn't there either:

```python
from bw_timex import TimexLCA, TimexLCASettings

demand = {("foreground", "driving"): 1}
method = ("ecoinvent-3.12", "EF v3.1", "climate change", "global warming potential (GWP100)")

tlca = TimexLCA(
    demand=demand,
    method=method,
    scenario={
        "iam_model": "remind",
        "pathway": "SSP2-NDC",
        "system_model": "cutoff",
        "ecoinvent_version": "3.12",
        "years": [2020, 2030, 2040, 2050, 2075, 2100],
    },
    create_missing=True,
    premise_key="dummy_premise_decryption_key",              # or $PREMISE_KEY
    ecoinvent_credentials=("dummy_user", "dummy_password"),  # or $ECOINVENT_USERNAME / _PASSWORD
)
```

Starting from an empty project, that reports what it is about to do and then
hands over to premise:

```text
INFO | No database 'ecoinvent-3.12-cutoff' in this project. Importing ecoinvent
       3.12 (cutoff) first; this takes a while and needs an ecoinvent licence.
INFO | Building 6 background database(s) for year(s) [2020, 2030, 2040, 2050, 2075, 2100]
       with premise (remind, SSP2-NDC, all sectors). Each is a full copy of ecoinvent,
       so expect tens of minutes and roughly 2-4 GB per year.
```

Run the same code again and it builds nothing, because the databases are now
there and carry the metadata that makes them findable:

```text
INFO | All 6 requested background vintage(s) already exist in this project.
       Nothing to build.
```

That metadata - `representative_time` plus the scenario keys, written by
premise >= 2.4.9.2 - is the whole reason nothing below has to map databases to
points in time by hand.

## Setting up

Picking up with those databases in place. The one thing `bw_timex` cannot
read from metadata is which database is the foreground, since that is a
property of your model rather than of the data:

In [1]:
from datetime import datetime

import bw2data as bd

from bw_timex import TimexLCA, TimexLCASettings, set_database_metadata

bd.projects.set_current("ei312_REMIND_EU")
set_database_metadata("foreground", representative_time="dynamic")

demand = {("foreground", "driving"): 1}  # one EV over its lifetime
method = ("ecoinvent-3.12", "EF v3.1", "climate change", "global warming potential (GWP100)")

## One calculation: `TimexLCASettings` and `run()`

In [18]:
TimexLCA(settings).run()

2026-08-24 11:52:07.954 | INFO     | bw_timex.timex_lca:__init__:421 - Initializing TimexLCA object...
2026-08-24 11:52:07.971 | INFO     | bw_timex.timex_lca:__init__:508 - Calculating base LCA...
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.90e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-24 11:52:08.788 | INFO     | bw_timex.timex_lca:__init__:525 - Collecting node infos...
2026-08-24 11:52:08.835 | INFO     | bw_timex.timex_lca:__init__:537 - Loading node metadata from 10 database(s)...
2026-08-24 11:52:09.003 | INFO     | bw_timex.timex_lca:__init__:574 - TimexLCA initialized.
2026-08-24 11:52:09.004 | INFO     | bw_timex.timex_lca:run:738 - Starting TimexLCA.run() pipeline...
2026-08-24 11:52:09.005 | INFO     | bw_timex.timex_lca:run:745 - Step 1/4: Building timeline...
2026-08-24 11:52:09.009 | INFO     | bw_timex.timex_lca:build_timeline:1

KeyboardInterrupt: 

In [19]:

settings = TimexLCASettings(
    demand=demand,
    method=method,    
    scenario={
        "iam_model": "remind",
        "pathway": "SSP2-NDC",
        "system_model": "cutoff",
        "ecoinvent_version": "3.12",
        "years": [2020, 2030, 2040, 2050, 2075, 2100],
    },
    timeline={"starting_datetime": datetime(2020, 6, 1), "graph_traversal": "bfs"},  # bought in 2020, driven from then on
    lci={
        "build_dynamic_biosphere": False,           # only the static score is needed here
    },
    lcia={
        "dynamic_lcia_enabled": False
    },
    label="bought 2020",
    
)

tlca = TimexLCA(settings).run()
print("base score:  ", round(tlca.base_score))
print("static score:", round(tlca.static_score))

2026-08-24 11:52:25.188 | INFO     | bw_timex.timex_lca:__init__:421 - Initializing TimexLCA object...


ValueError: No database in this project declares the metadata key(s) ['ecoinvent_version', 'iam_model', 'pathway', 'system_model']. Keys declared by the databases of this project: overwrite, read_only, representative_time. Add the metadata with `bw_timex.set_database_metadata`, check the spelling of your `scenario` filter, or build the databases with premise - `bw_timex.ensure_scenario_databases(scenario)`, or `TimexLCA(..., create_missing=True)`, both with a `years` list in the scenario.

The time-explicit score is higher than the base score because the EV's
30,000 kWh are spread over its lifetime and resolved against the REMIND-EU
vintage each year of use falls into, rather than the 2020 database the
exchange nominally points at.

The traversal's score-coverage warning is expected: it only measures the
*foreground* graph it walks, while most of this system's impact sits in the
background, which is resolved by solving the expanded matrix instead. The
score is unchanged across a wide range of `cutoff` values.

`run()` again on the same object, changing only what should change -
the settings object itself is left untouched, and the base LCA is reused:

In [4]:
base_lca_id = id(tlca.base_lca)

tlca.run(starting_datetime=datetime(2075, 6, 1))  # same EV, bought decades later
print("static score, bought 2075:", round(tlca.static_score))
print("base LCA reused:", id(tlca.base_lca) == base_lca_id)

2026-08-24 09:46:12.251 | INFO     | bw_timex.timex_lca:run:738 - Starting TimexLCA.run() pipeline...
2026-08-24 09:46:12.254 | INFO     | bw_timex.timex_lca:run:745 - Step 1/4: Building timeline...
2026-08-24 09:46:12.258 | INFO     | bw_timex.timex_lca:run:760 - Step 2/4: Calculating LCI...
2026-08-24 09:46:12.982 | INFO     | bw_timex.timex_lca:lci:1350 - Expanding matrices...
2026-08-24 09:46:13.068 | INFO     | bw_timex.timex_lca:lci:1369 - Calculating dynamic inventory...
2026-08-24 09:46:17.519 | INFO     | bw_timex.timex_lca:run:769 - Step 3/4: Calculating static LCIA...
2026-08-24 09:46:17.526 | INFO     | bw_timex.timex_lca:run:776 - Step 4/4: Skipping dynamic LCIA (disabled).
2026-08-24 09:46:17.527 | INFO     | bw_timex.timex_lca:run:799 - TimexLCA.run() completed successfully.


static score, bought 2075: 8375
base LCA reused: True


A third of the 2020 purchase's impact, on the same EV - that is REMIND-EU's grid decarbonizing under the vehicle.

## Several calculations: `compare()`

`compare()` takes a list of settings and returns a `ComparisonResult`
whose `summary` holds one row each - the scores next to every setting that
produced them, so the table is its own record of what was run. It builds one
`TimexLCA` per distinct background, so the purchase years below share a
single object:

In [5]:
from dataclasses import replace

comparison = TimexLCA.compare(
    [
        replace(settings, starting_datetime=datetime(year, 6, 1), label=f"bought {year}")
        for year in (2020, 2040, 2075)
    ]
)


2026-08-24 09:46:44.216 | INFO     | bw_timex.timex_lca:__init__:421 - Initializing TimexLCA object...
2026-08-24 09:46:44.224 | INFO     | bw_timex.timex_lca:__init__:508 - Calculating base LCA...
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.90e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-24 09:46:45.218 | INFO     | bw_timex.timex_lca:__init__:525 - Collecting node infos...
2026-08-24 09:46:45.274 | INFO     | bw_timex.timex_lca:__init__:537 - Loading node metadata from 10 database(s)...
2026-08-24 09:46:45.541 | INFO     | bw_timex.timex_lca:__init__:574 - TimexLCA initialized.
2026-08-24 09:46:45.541 | INFO     | bw_timex.timex_lca:compare:948 - Comparison 1/3: bought 2020
2026-08-24 09:46:45.542 | INFO     | bw_timex.timex_lca:run:738 - Starting TimexLCA.run() pipeline...
2026-08-24 09:46:45.542 | INFO     | bw_timex.timex_lca:run:745 - Step 

Starting graph traversal


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/bw_graph_tools/graph_traversal/new_node_each_visit.py:351: UserWarning: Graph traversal covered only 0.2% of the total LCA score. Consider lowering the `cutoff` (currently 1e-09) to improve coverage.
  warnings.warn(
2026-08-24 09:47:15.143 | INFO     | bw_timex.timeline_builder:build_timeline:183 - Building timeline...


Calculation count: 103


2026-08-24 09:47:15.362 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:808 - Reference date 2018-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-08-24 09:47:15.476 | INFO     | bw_timex.timex_lca:run:760 - Step 2/4: Calculating LCI...
2026-08-24 09:47:15.985 | INFO     | bw_timex.timex_lca:lci:1350 - Expanding matrices...
2026-08-24 09:47:16.010 | INFO     | bw_timex.timex_lca:lci:1369 - Calculating dynamic inventory...
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 1.65e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-24 09:47:19.042 | INFO     | bw_timex.timex_lca:run:769 - Step 3/4: Calculating static LCIA...
2026-08-24 09:47:19.047 | INFO     | bw_timex.timex_lca:run:776 - Step 4/4: Skipping dynamic LCIA (disabled).
2026-08-24 09:47:19.048 | INFO     | b

...

Starting graph traversal


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/bw_graph_tools/graph_traversal/new_node_each_visit.py:351: UserWarning: Graph traversal covered only 0.2% of the total LCA score. Consider lowering the `cutoff` (currently 1e-09) to improve coverage.
  warnings.warn(
2026-08-24 09:47:34.311 | INFO     | bw_timex.timeline_builder:build_timeline:183 - Building timeline...


Calculation count: 103


2026-08-24 09:47:34.506 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:815 - Reference date 2073-01-01 00:00:00 is higher than all provided dates. Data will be taken from the closest lower year.
2026-08-24 09:47:34.568 | INFO     | bw_timex.timex_lca:run:760 - Step 2/4: Calculating LCI...
2026-08-24 09:47:35.189 | INFO     | bw_timex.timex_lca:lci:1350 - Expanding matrices...
2026-08-24 09:47:35.281 | INFO     | bw_timex.timex_lca:lci:1369 - Calculating dynamic inventory...
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 2.10e+13)
  warnings.warn(msg, UmfpackWarning)
2026-08-24 09:47:39.938 | INFO     | bw_timex.timex_lca:run:769 - Step 3/4: Calculating static LCIA...
2026-08-24 09:47:39.944 | INFO     | bw_timex.timex_lca:run:776 - Step 4/4: Skipping dynamic LCIA (disabled).
2026-08-24 09:47:39.944 | INFO     | b

,label,base_score,static_score,timeline_rows,runtime_s
0,bought 2020,23247.151124,25462.997441,27,33.510305
1,bought 2040,23247.151124,9127.954269,27,10.905721
2,bought 2075,23247.151124,8375.067004,27,9.985533


In [9]:
comparison.summary[["label", "base_score", "static_score", "timeline_rows", "runtime_s"]]

,label,base_score,static_score,timeline_rows,runtime_s
0,bought 2020,23247.151124,25462.997441,27,33.510305
1,bought 2040,23247.151124,9127.954269,27,10.905721
2,bought 2075,23247.151124,8375.067004,27,9.985533


Two options worth knowing: `keep_objects=True` keeps each `TimexLCA`
in `ComparisonResult.objects`, to dig into one result's timeline or dynamic
inventory afterwards; `on_error="record"` puts a failure in the row's `error`
column and carries on, instead of aborting a long unattended sweep.

Comparing *scenarios* rather than purchase years works the same way - give
each settings object a different `scenario={...}` (say `SSP2-NDC` against
`SSP2-PkBudg500`). Each distinct background gets its own `TimexLCA`, since the
background fixes the columns of the time-explicit matrices; that is also why
`run()` refuses a `scenario` change on an existing object and points here
instead.